# SNR Evaluation of ICA

**Dataset**: PhysioNet Auditory EEG  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

We compute the signal-to-noise ratio (SNR) for all 4 channels before and after ICA cleaning. We define signal as alpha band power (8-13 Hz) and noise as power outside it.

## Expected outputs

- Grouped bar chart with 4 channel groups
- Each group has two bars: before ICA (blue) and after (orange)
- Higher SNR after cleaning indicates effectiveness

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Band | 8-13 Hz | Alpha band |
| nperseg | 1024 | Welch window |
| n_components | 4 | ICA components |


## 1. Install dependencies


In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2.


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


## 4. Compute SNR before and after cleaning

We apply ICA, exclude the highest-variance component, then compute SNR for each channel before and after cleaning.


In [ ]:
import mne
from scipy.signal import welch

info = mne.create_info(ch_names, sfreq=fs, ch_types='eeg')
raw = mne.io.RawArray(eeg_data.T * 1e-6, info, verbose=False)

# NOTE: ICA works best with more channels than components.
ica = mne.preprocessing.ICA(
    n_components=3, random_state=97, max_iter=800, verbose=False
)
ica.fit(raw, verbose=False)

component_variances = np.var(ica.get_sources(raw).get_data(), axis=1)
exclude_idx = int(np.argmax(component_variances))
ica.exclude = [exclude_idx]
cleaned_raw = ica.apply(raw.copy(), verbose=False)
cleaned_data = cleaned_raw.get_data() * 1e6

def compute_snr(data, fs, fmin=8, fmax=13):
    freqs, psd = welch(data, fs=fs, nperseg=1024)
    signal_mask = (freqs >= fmin) & (freqs <= fmax)
    noise_mask = (freqs >= 0.5) & (freqs <= 80) & ~signal_mask
    signal_power = np.trapezoid(psd[signal_mask], freqs[signal_mask])
    noise_power = np.trapezoid(psd[noise_mask], freqs[noise_mask])
    if noise_power == 0:
        return 0.0
    return 10 * np.log10(signal_power / noise_power)

snr_before = [compute_snr(eeg_data[:, i], fs) for i in range(4)]
snr_after = [compute_snr(cleaned_data[i], fs) for i in range(4)]
for i, name in enumerate(ch_names):
    print(f'{name}: Before={snr_before[i]:.2f} dB, After={snr_after[i]:.2f} dB')


## 5. Interactive plot

**What to look for:**

- Each channel has two bars: before (blue) and after (orange) ICA
- Higher orange bar indicates SNR improvement
- More affected channels benefit more



In [ ]:
import plotly.graph_objects as go

x = list(ch_names)
fig = go.Figure()
fig.add_trace(go.Bar(name='Before ICA', x=x, y=snr_before,
                     marker_color='steelblue'))
fig.add_trace(go.Bar(name='After ICA', x=x, y=snr_after,
                     marker_color='orange'))
fig.update_layout(barmode='group', height=500,
                  title='SNR Evaluation - Before vs After ICA Cleaning',
                  xaxis_title='Channel', yaxis_title='SNR (dB)')
fig.show()


## What did we learn?

- SNR is a quantitative measure comparing signal power to noise power
- Higher SNR after cleaning indicates effectiveness
- The definition of signal and noise affects the values
- Complements visual evaluation via FFT

